In [1]:
import os
import math
import time
import torch
import pickle
import numpy as np

import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.autograd as autograd

import torch.nn.functional as functional
import torch.optim.lr_scheduler as lr_scheduler
from utils import *

In [2]:
os.makedirs('./results', exist_ok=True)
torch.set_default_dtype(torch.float64) 
_ = torch.manual_seed(0)

In [3]:
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [4]:
if torch.cuda.is_available():
    torch_device = 'cuda'
else:
    torch_device = 'cpu'
print(f"TORCH DEVICE: {torch_device}")

TORCH DEVICE: cuda


In [5]:
# Multivariate gaussian
mean = [-1.5,1.5]
mean = torch.tensor(mean,device=torch_device)
# cov  = torch.eye(2).to(device = torch_device)
cov = torch.tensor([[1.0,0.5],[0.5,1.0]],device=torch_device)


In [6]:
# def log_prob(x,mean = mean, scale = 1.):
#     x.requires_grad_(True)
#     logp = -1.0*torch.log(torch.tensor(2*np.pi)) - ((x[:, 0] - mean[0]) ** 2 + (x[:, 1] - mean[1]) ** 2)/(2 * scale**2)
#     return logp

In [ ]:
# def log_prob(x,mean = mean,cov=cov):
#     x.requires_grad_(True)
#     logp = -1.0*torch.log(torch.tensor(2*np.pi)) - 0.5*(x-mean).T*torch.linalg.inv(cov)*(x-mean)
#     return logp

In [7]:
def grad_dist(x):
    x = x.cpu().detach()
    y1 = -(2/3)*(2*x[:,0]-x[:,1]+4.5)
    y2 = -(2/3)*(2*x[:,1]-x[:,0]-4.5)
    return torch.tensor(np.array([y1,y2]).T)

In [ ]:
# def grad_dist(x):
#     x = x.cpu().detach()
#     y1 = -(x[:,0] + 1.5)
#     y2 = -(x[:,1] - 1.5)
#     return torch.tensor(np.array([y1,y2]).T)

In [8]:
batch_size = 512
no_training_samples = 50000
no_validation_samples = 5000
no_testing_samples = 5000

dist  = torch.distributions.MultivariateNormal(mean, covariance_matrix = cov,validate_args=False)
data  = dist.sample((no_training_samples + no_validation_samples + no_testing_samples,)).to(device=torch_device)

train_dataloader = torch.utils.data.DataLoader(data[:no_training_samples], batch_size = batch_size, shuffle=True)
val_dataloader = torch.utils.data.DataLoader(data[no_training_samples:no_training_samples + no_validation_samples], batch_size = batch_size, shuffle=False)
test_dataloader = torch.utils.data.DataLoader(data[no_training_samples + no_validation_samples:], batch_size = batch_size, shuffle= False)

In [9]:
actual_grad =  - grad_dist(data[:50000].cpu())

In [10]:
actual_grad.shape

torch.Size([50000, 2])

In [11]:
array = [data[:50000].cpu().detach().numpy(), actual_grad.numpy()]

In [12]:
output = open("Training_data_multivariate_for_one_gaussian_diag_variance_with_neg_score_through_exact_pdf_function.pkl", 'wb')
pickle.dump(array, output)
output.close()

In [13]:
model = mlp(sizes=[2, 256, 256, 2]).to(device = torch_device)


In [18]:
PATH = "./results/score_estimation_model_with_fixed_mean_diag_cov.pt"

In [19]:
#For resuming the checkpoint:
checkpoint = torch.load(PATH)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

In [15]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
for epoch in range(250):
    loss = score_matching(model, data[:no_training_samples])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()    
    test_loss = score_matching(model,data[no_training_samples + no_validation_samples:])
    print(f"Epoch: {(epoch+1):3d} , loss: {loss:.4f} , test loss: {test_loss:.4f}")



In [ ]:
epochs = 50

for i in range(epochs):
    model.train()
    train_score_loss = 0.
    val_score_loss = 0.
    
    for x in train_dataloader:
        loss = score_matching(model, x)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_score_loss += loss.item()*(x.shape[0])
        
  
    # model.eval()
    # with torch.no_grad():
    for x in val_dataloader:
        val_loss = score_matching(model, x)
        val_score_loss += val_loss.item()*(x.shape[0])
            
    avg_train_loss = train_score_loss/no_training_samples
    avg_val_loss   = val_score_loss/no_validation_samples
            
    template = 'Epoch {:3d}, Train loss: {:.4f},Val loss: {:.4f}'
    print (template.format(i+1 , avg_train_loss , avg_val_loss))
    

In [ ]:
#model.eval()
test_loss = score_matching(model,data[no_training_samples + no_validation_samples:])

print(f'Test Loss: {test_loss:.4f}')

In [ ]:
score_matching(model,data[:no_training_samples])

In [ ]:
torch.save({'model_state_dict': model.state_dict(),'optimizer_state_dict': optimizer.state_dict(),}, PATH)

In [ ]:
model(data[:5])

In [ ]:
data[:5]

In [ ]:
#jacobian1(dist.log_prob,data[:5])
grad_dist(data[:5])

In [ ]:
data[:5].shape

In [ ]:
filename = "multivariate_data_for_one_gaussian_with_neg_logp.pkl"
f = open(filename, 'rb')
dataset = pickle.load(f,encoding="latin1")

In [ ]:
dataset[0].shape

In [ ]:
data_gauss = torch.tensor(dataset[0] , dtype = torch.float64 , device = torch_device)

In [ ]:
data_gauss.shape

In [20]:
neg_score = - model(data[:50000])

In [21]:
neg_score.shape

torch.Size([50000, 2])

In [22]:
arr = [data[:50000].cpu().detach(),neg_score.cpu().detach()]

In [23]:
output = open("Training_data_multivariate_data_for_one_gaussian_diag_varaiance_with_neg_score_through_score_model.pkl", 'wb')
pickle.dump(arr, output)
output.close()

In [ ]:
actual_neg_score = -grad_dist(data_gauss)

In [ ]:
neg_score.shape

In [ ]:
actual_neg_score.shape

In [ ]:
array_1 = [data_gauss.cpu().detach(),actual_neg_score.cpu().detach()]

In [ ]:
output = open("multivariate_data_for_one_gaussian_with_neg_score_through_pdf_function.pkl", 'wb')
pickle.dump(array_1, output)
output.close()